# Test d'intégration Batch

Ce notebook sert à tester automatiquement l'ensemble du pipeline en mode batch.

Le principe général est :

`Nettoyage → Initialisation → Payload 1 → Pipeline → Validation → Payload 2 → Pipeline → Validation`

L'objectif est de vérifier que les données passent correctement dans :

`Landing Zone → Bronze → Silver → Gold`

et que le pipeline fonctionne aussi lorsqu'un deuxième lot de données arrive.

In [0]:
dbutils.widgets.text(
    "Environment",
    "dev",
    "Set the current environment/catalog name"
)

env = dbutils.widgets.get("Environment")

## 1. Choisir l'environnement

Le widget `Environment` permet de choisir le catalogue dans lequel le test sera exécuté.

Dans notre cas :

`Environment = dev`

La valeur est ensuite récupérée dans la variable Python `env`.

Cela permet de réutiliser le même notebook plus tard avec un autre environnement comme `test` ou `prod` sans modifier le code.

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/02-setup

## 2. Charger le Setup

Le notebook `02-setup` contient la classe `SetupHelper`.

Cette classe est nécessaire pour créer, vérifier ou nettoyer les objets du projet :

`Bronze + Silver + Gold`

Le `%run` charge donc la définition de `SetupHelper` dans le notebook de test afin que nous puissions l'utiliser ensuite.

In [0]:
SH = SetupHelper(env)
SH.cleanup()

## 3. Nettoyer l'environnement

On crée un objet :

`SH = SetupHelper(env)`

puis on exécute :

`SH.cleanup()`

Le but est de repartir d'un environnement propre avant le test.

Cela évite que des tables ou des données provenant d'un ancien test faussent les résultats.

Attention : le cleanup des tables Databricks ne signifie pas forcément que les fichiers présents dans la landing zone Azure `raw` sont supprimés. Pour un test complètement propre, il faut également vérifier que les anciens fichiers de test ne sont plus présents dans `raw`.

In [0]:
dbutils.notebook.run(
    "/Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/07-run",
    600,
    {
        "Environment": env,
        "RunType": "once"
    }
)

## 4. Premier lancement de `07-run`

On appelle ensuite le notebook principal :

`07-run`

avec :

`Environment = dev`

et :

`RunType = once`

Le mode `once` signifie que les streams utilisent le principe `availableNow` :

`traiter toutes les nouvelles données disponibles → puis s'arrêter`

Ce premier lancement sert surtout à préparer l'environnement après le cleanup :

`Setup → History Loader → Bronze → Silver → Gold`

À ce stade, normalement aucun Payload n'a encore été produit. Si la landing zone `raw` est vide, Bronze, Silver et Gold n'ont donc aucune nouvelle donnée métier à traiter.

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/03-history-loader

## 5. Charger et vérifier le History Loader

Le notebook `03-history-loader` contient la classe `HistoryLoader`.

Il permet de charger les données historiques ou de référence nécessaires au projet.

Dans notre cas, il sert principalement à charger :

`date_lookup`

Cette table calendrier est utilisée ensuite par le pipeline, notamment pour enrichir certaines données.

Les commandes :

`SH.validate()`

et :

`HL.validate()`

permettent de vérifier que l'environnement et les données historiques ont bien été initialisés avant de continuer.

In [0]:
HL = HistoryLoader(env)

SH.validate()
HL.validate()

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/10-producer

## 6. Charger le Producer

Le notebook `10-producer` contient la classe `Producer`.

Le Producer ne transforme aucune donnée.

Son rôle est de simuler l'arrivée de nouvelles données dans le système.

Les fichiers existent d'abord dans :

`data_zone/test_data`

puis le Producer les copie vers :

`data_zone/raw`

Le flux est donc :

`test_data → Producer → raw → Bronze`

In [0]:
# Code pour regler le probleme de AssertionError:  
conf = Config()

raw_dir = conf.base_dir_data + "/raw"
checkpoint_dir = conf.base_dir_checkpoint + "/checkpoints"

print("Cleaning RAW...")
dbutils.fs.rm(raw_dir, True)

print("Cleaning checkpoints...")
dbutils.fs.rm(checkpoint_dir, True)

# Recréation des dossiers nécessaires au Producer
dbutils.fs.mkdirs(raw_dir + "/registered_users_bz")
dbutils.fs.mkdirs(raw_dir + "/gym_logins_bz")
dbutils.fs.mkdirs(raw_dir + "/kafka_multiplex_bz")

print("RAW and checkpoints cleaned.")

In [0]:
display(
    dbutils.fs.ls(
        conf.base_dir_data + "/raw/registered_users_bz"
    )
)

In [0]:
PR = Producer()

PR.produce(1)
PR.validate(1)

## 7. Produire le Payload 1

On crée :

`PR = Producer()`

puis :

`PR.produce(1)`

Le Producer copie le premier jeu de données dans la landing zone.

Le Payload 1 contient notamment :

- les inscriptions utilisateurs ;
- les événements de profil CDC ;
- les mesures BPM ;
- les événements Workout ;
- les entrées/sorties Gym.

La commande :

`PR.validate(1)`

vérifie ensuite que les fichiers ont bien été copiés et que le nombre de lignes correspond aux valeurs attendues.

À ce stade, les données sont seulement disponibles dans `raw`. Elles ne sont pas encore transformées.

In [0]:
dbutils.notebook.run(
    "/Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/07-run",
    600,
    {
        "Environment": env,
        "RunType": "once"
    }
)

## 8. Relancer `07-run` pour le Payload 1

Une fois le Payload 1 présent dans la landing zone, on relance :

`07-run`

Le pipeline peut maintenant réellement traiter les données.

Le flux devient :

`Payload 1 → raw → Bronze → Silver → Gold`

Bronze ingère les fichiers bruts.

Silver nettoie, déduplique, traite le CDC et construit les tables intermédiaires.

Gold calcule les résultats métier comme `workout_bpm_summary`.

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/04-bronze

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/05-silver

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/06-gold

## 9. Valider Bronze, Silver et Gold pour le Payload 1

On charge les classes :

`Bronze`

`Silver`

`Gold`

puis on crée les objets :

`BZ = Bronze(env)`

`SL = Silver(env)`

`GL = Gold(env)`

Les commandes :

`BZ.validate(1)`

`SL.validate(1)`

`GL.validate(1)`

vérifient que chaque couche contient les résultats attendus après le premier lot de données.

On ne vérifie donc pas seulement que le pipeline s'est exécuté sans erreur : on vérifie aussi que les données produites sont correctes.

In [0]:
BZ = Bronze(env)
SL = Silver(env)
GL = Gold(env)

BZ.validate(1)
SL.validate(1)
GL.validate(1)

In [0]:
PR.produce(2)
PR.validate(2)

## 10. Produire le Payload 2

On exécute ensuite :

`PR.produce(2)`

Le Payload 2 représente un nouveau lot de données arrivé après le premier traitement.

Il peut contenir :

- de nouveaux utilisateurs ;
- de nouveaux événements ;
- de nouvelles mesures BPM ;
- de nouvelles séances ;
- des mises à jour CDC d'utilisateurs existants.

Le but est de vérifier que le pipeline sait fonctionner de manière incrémentale.

`PR.validate(2)` vérifie que les deux lots présents dans la landing zone correspondent maintenant aux quantités attendues.

In [0]:
dbutils.notebook.run(
    "/Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/07-run",
    600,
    {
        "Environment": env,
        "RunType": "once"
    }
)

## 11. Relancer `07-run` pour le Payload 2

Le pipeline est exécuté une nouvelle fois.

Grâce aux checkpoints et aux traitements incrémentaux, il doit traiter uniquement ce qui n'a pas encore été traité.

Le principe est :

`Résultat Payload 1 + nouvelles données Payload 2 → nouvel état Bronze/Silver/Gold`

Cette étape permet notamment de vérifier :

- le fonctionnement des checkpoints ;
- l'absence de doublons ;
- les MERGE ;
- les mises à jour CDC ;
- l'évolution correcte des résultats Gold.

In [0]:
BZ.validate(2)
SL.validate(2)
GL.validate(2)

## 12. Validation finale avec le Payload 2

On exécute :

`BZ.validate(2)`

`SL.validate(2)`

`GL.validate(2)`

Ces validations vérifient l'état final après les deux lots.

Le but est de confirmer que :

`Payload 1 + Payload 2`

ont été correctement intégrés dans toutes les couches du Lakehouse.

## Résumé du scénario Batch

Le test complet suit donc cette logique :

`Cleanup`

`↓`

`07-run initial → Setup + History`

`↓`

`Payload 1`

`↓`

`07-run`

`↓`

`Validation Bronze / Silver / Gold`

`↓`

`Payload 2`

`↓`

`07-run`

`↓`

`Validation Bronze / Silver / Gold`

`↓`

`Cleanup`

Ce test est un test d'intégration end-to-end : il vérifie que tous les composants du projet fonctionnent correctement ensemble, depuis l'arrivée des fichiers jusqu'aux résultats finaux Gold.